> Agent! Agent! Agent! = multi-turn trajectory

- 对于 Agent 重要、基础的能力
    - coding
    - search
- Agentic tuning
    - 训练的是：tool use grounding 的能力

### new special tokens


- DeepSeek V3.1 add four new special tokens (compared to V3-base)
    - `<｜search▁begin｜>` (id: 128796)
    - `<｜search▁end｜>` (id: 128797)
        - 网页端默认开启搜索，既使为选中搜索按钮
        - 目前官方的 huggingface 版本，在 chat template 中没有关于这对 special token 的处理
    - `<think>` (id: 128798)
    - `</think>` (id: 128799)
        - 之前只有 R1 模型才有这个 special token

| Token | 功能 (Function) |
| :--- | :--- |
| `<｜begin▁of▁sentence｜>` | 整个输入的起始符。`bos_token` |
| `<｜end▁of▁sentence｜>` | 标记一轮完整的 Assistant 回复或工具交互的结束。 |
| `<｜User｜>` | 标记用户输入的开始。 |
| `<｜Assistant｜>` | 标记模型回复的开始。 |
| `<think></think>` | 区分模型的两种工作模式。`<think>` 开启思考模式，`</think>` 开启非思考工具调用模式。 |
| `<｜tool▁calls▁begin/end｜>` | 包裹一个或多个工具调用的代码块。 |
| `<｜tool▁call▁begin/end｜>` | 包裹单个工具调用的代码块。 |
| `<｜tool▁sep｜>` | 在工具名和其 JSON 参数之间进行分隔。 |
| `<｜tool▁output▁begin/end｜>` | 包裹工具执行后返回的输出。 |
| `<｜search▁begin/end｜>` | Search-Agent 模式下，用于包裹整个“思考-搜索-整合”过程的特殊 Token。 |

In [2]:
import transformers

tokenizer = transformers.AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3.1")

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Who are you?"},
    {"role": "assistant", "content": "<think>Hmm</think>I am DeepSeek"},
    {"role": "user", "content": "1+1=?"}
]

In [3]:
tokenizer.apply_chat_template(messages, tokenize=False, thinking=True, add_generation_prompt=True)

'<｜begin▁of▁sentence｜>You are a helpful assistant<｜User｜>Who are you?<｜Assistant｜></think>I am DeepSeek<｜end▁of▁sentence｜><｜User｜>1+1=?<｜Assistant｜><think>'

In [4]:
tokenizer.apply_chat_template(messages, tokenize=False, thinking=False, add_generation_prompt=True)

'<｜begin▁of▁sentence｜>You are a helpful assistant<｜User｜>Who are you?<｜Assistant｜></think>I am DeepSeek<｜end▁of▁sentence｜><｜User｜>1+1=?<｜Assistant｜></think>'

#### tool calls

- Toolcall is supported in non-thinking mode. The format is:

```jinja
{%- if message['role'] == 'assistant' and message['tool_calls'] is defined and message['tool_calls'] is not none %}
{%- if ns.is_last_user %}
  {{'<｜Assistant｜></think>'}}
```

- Assistant 的工具调用行动，以 `<｜tool▁call▁end｜><｜end▁of▁sentence｜>` 为结束和收尾，后续进入 `message['role'] == 'tool'` 的处理

```jinja
...
    {%- for tool in message['tool_calls'] %}
      ... (代码省略，用于格式化每个tool call) ...
    {%- endfor %}
    {{'<｜tool▁call▁end｜><｜end▁of▁sentence｜>'}}  <-- 在这里！
...
```

#### `<｜search▁begin/end｜>` 

- scope 非常大，且没有被 chat template 管理
- 像是一个完整的 ReAct，一个更大的事务块（Agentic）的概念？
    - reasoning→ [tool calls(search/python)→观察(tool resp)]*n →answer

### trajectory 分析

- Code-Agent
- Search-Agent
    - search-tool
    - search-python-tool
        - hle 的一道题目